# MultiIndex / advanced indexing

### Creating a MultiIndex (hierarchical index) object
The **`MultiIndex`** object is the hierarchical analogue of the standard **`Index`** object which typically stores the axis labels in pandas objects. You can think of `MultiIndex` as an array of tuples where each tuple is unique. A `MultiIndex` can be created from a list of arrays (using **`MultiIndex.from_arrays()`**), an array of tuples (using **`MultiIndex.from_tuples()`**), a crossed set of iterables (using **`MultiIndex.from_product()`**), or a **`DataFrame`** (using **`MultiIndex.from_frame()`**). The `Index` constructor will attempt to return a `MultiIndex` when it is passed a list of tuples. The following examples demonstrate different ways to initialize MultiIndexes.


In [1]:
import pandas as pd
import numpy as np

In [2]:
arrays = [
    ["bar", "bar", "baz", "baz", "foo", "foo", "qux", "qux"],
    ["one", "two", "one", "two", "one", "two", "one", "two"],
]

In [3]:
tuples = list(zip(*arrays))

In [4]:
tuples

[('bar', 'one'),
 ('bar', 'two'),
 ('baz', 'one'),
 ('baz', 'two'),
 ('foo', 'one'),
 ('foo', 'two'),
 ('qux', 'one'),
 ('qux', 'two')]

In [5]:
index = pd.MultiIndex.from_tuples(tuples, names = ["first", "second"])

In [6]:
index

MultiIndex([('bar', 'one'),
            ('bar', 'two'),
            ('baz', 'one'),
            ('baz', 'two'),
            ('foo', 'one'),
            ('foo', 'two'),
            ('qux', 'one'),
            ('qux', 'two')],
           names=['first', 'second'])

In [7]:
s = pd.Series(np.random.randn(8), index = index)

In [8]:
s

first  second
bar    one      -0.477340
       two      -1.027425
baz    one      -2.094849
       two      -1.215478
foo    one      -0.153792
       two       1.473840
qux    one       1.849406
       two      -1.560311
dtype: float64

In [9]:
# When you want every pairing of the elements in two iterables, it can be easier to use the MultiIndex.from_product() method:
iterables = [['bar', 'baz', 'foo', 'qux'], ['one', 'two']]
pd.MultiIndex.from_product(iterables, names = ['first', 'second'])

MultiIndex([('bar', 'one'),
            ('bar', 'two'),
            ('baz', 'one'),
            ('baz', 'two'),
            ('foo', 'one'),
            ('foo', 'two'),
            ('qux', 'one'),
            ('qux', 'two')],
           names=['first', 'second'])

In [10]:
# You can also construct a MultiIndex from a DataFrame directly, using the method MultiIndex.from_frame(). This is a complementary method to MultiIndex.to_frame().
df = pd.DataFrame(
    [['bar', 'one'], ['bar', 'two'], ['foo', 'one'], ['foo', 'two']],
    columns = ['first', 'second'],
)

pd.MultiIndex.from_frame(df)

MultiIndex([('bar', 'one'),
            ('bar', 'two'),
            ('foo', 'one'),
            ('foo', 'two')],
           names=['first', 'second'])

In [11]:
# As a convenience, you can pass a list of arrays directly into Series or DataFrame to construct a MultiIndex automatically:
arrays = [
    np.array(["bar", "bar", "baz", "baz", "foo", "foo", "qux", "qux"]),
    np.array(["one", "two", "one", "two", "one", "two", "one", "two"]),
]

s = pd.Series(np.random.randn(8), index = arrays)
s

bar  one    0.756310
     two    0.279375
baz  one    0.434092
     two   -1.363893
foo  one   -0.641515
     two   -0.385203
qux  one   -2.209597
     two   -0.757931
dtype: float64

In [12]:
df = pd.DataFrame(np.random.randn(8, 4), index=arrays)
df

0         1         2         3
bar one  1.021509  0.178411  0.084789  0.435101
    two  2.434275  0.504740 -0.092543 -0.123579
baz one -0.755443 -1.108774 -0.459126 -0.755987
    two -0.091931  2.352552 -1.352234  1.175758
foo one -1.324455 -0.060347  0.288853  1.500534
    two  0.271870  1.267957  0.676972  0.324581
qux one  1.205295  1.564493 -0.588332  0.914346
    two  2.058651  1.200806  0.035453  2.066059

In [13]:
# All of the MultiIndex constructors accept a names argument which stores string names for the levels themselves. If no names are provided, None will be assigned:
df.index.names

FrozenList([None, None])

In [14]:
# This index can back any axis of a pandas object, and the number of levels of the index is up to you:

df = pd.DataFrame(np.random.randn(3, 8), index = ["A", "B", "C"], columns = index)
df

first        bar                 baz                 foo                 qux  \
second       one       two       one       two       one       two       one   
A       1.298022  0.227811  0.158003  0.940212 -0.231041 -1.076115 -0.014290   
B      -0.220813  1.271892  0.158113 -0.812318  1.888800 -0.243946 -1.414743   
C       0.830147 -1.181357 -0.015977  1.011322 -1.048584 -0.490976 -0.736137   

first             
second       two  
A      -0.373190  
B       1.701547  
C       1.297449

In [15]:
pd.DataFrame(np.random.randn(6, 6), index = index[:6], columns = index[:6])

first              bar                 baz                 foo          
second             one       two       one       two       one       two
first second                                                            
bar   one     0.158252  1.063370 -1.141980  0.681006  1.875805 -0.326804
      two     0.281413  1.294976  0.102975  0.428477 -0.210877  0.462114
baz   one     1.401601  0.707768 -0.848926 -1.102730  0.350009  0.475155
      two     0.408636 -2.068065  0.240462  0.925020 -1.943794 -0.237403
foo   one    -2.527152  1.189286 -0.399315  0.467779  0.206078 -2.524282
      two     0.029852  0.884171 -0.164750 -1.503995  0.512816  0.024970

In [16]:
# We’ve “sparsified” the higher levels of the indexes to make the console output a bit easier on the eyes. 
#Note that how the index is displayed can be controlled using the multi_sparse option in pandas.set_options():

with pd.option_context("display.multi_sparse", False):
    df

In [17]:
# It’s worth keeping in mind that there’s nothing preventing you from using tuples as atomic labels on an axis:
pd.Series(np.random.randn(8), index=tuples)

(bar, one)   -0.014284
(bar, two)    0.955270
(baz, one)   -1.199442
(baz, two)   -0.845838
(foo, one)    1.731066
(foo, two)   -1.327484
(qux, one)    0.681748
(qux, two)    1.027802
dtype: float64

The reason that the `MultiIndex` matters is that it can allow you to do grouping, selection, and reshaping operations as we will describe below and in subsequent areas of the documentation. As you will see in later sections, you can find yourself working with hierarchically-indexed data without creating a `MultiIndex` explicitly yourself. However, when loading data from a file, you may wish to generate your own `MultiIndex` when preparing the data set.


### Reconstructing the level labels
The method `get_level_values()` will return a vector of the labels for each location at a particular level:

In [18]:
index.get_level_values(0) 

Index(['bar', 'bar', 'baz', 'baz', 'foo', 'foo', 'qux', 'qux'], dtype='object', name='first')

In [19]:
index.get_level_values("second")

Index(['one', 'two', 'one', 'two', 'one', 'two', 'one', 'two'], dtype='object', name='second')

### Basic indexing on axis with MultiIndex

One of the important features of hierarchical indexing is that you can select data by a “partial” label identifying a subgroup in the data. **Partial** selection “drops” levels of the hierarchical index in the result in a completely analogous way to selecting a column in a regular DataFrame:


In [20]:
df['bar']

second,one,two
A,1.298022,0.227811
B,-0.220813,1.271892
C,0.830147,-1.181357


In [21]:
df["bar", "one"]

A    1.298022
B   -0.220813
C    0.830147
Name: (bar, one), dtype: float64

In [22]:
s["qux"]

one   -2.209597
two   -0.757931
dtype: float64

### Using slicers

You can slice a `MultiIndex` by providing multiple indexers.

You can provide any of the selectors as if you are indexing by label, including slices, lists of labels, labels, and boolean indexers.

You can use `slice(None)` to select all the contents of *that* level. You do not need to specify all the *deeper* levels, they will be implied as `slice(None)`.

As usual, **both sides** of the slicers are included as this is label indexing.

##### ⚠️ Warning

You should specify all axes in the `.loc` specifier, meaning the indexer for the **index** and for the **columns**. There are some ambiguous cases where the passed indexer could be misinterpreted as indexing *both* axes, rather than into say the `MultiIndex` for the rows.

You should do this:

```python
df.loc[(slice("A1", "A3"), ...), :]  # noqa: E999
```

You should **not** do this:

```python
df.loc[(slice("A1", "A3"), ...)]  # noqa: E999
```


In [23]:
def mklbl(prefix, n):
    return ["%s%s" % (prefix, i) for i in range(n)]


miindex = pd.MultiIndex.from_product(
    [mklbl("A", 4), mklbl("B", 2), mklbl("C", 4), mklbl("D", 2)]
)


micolumns = pd.MultiIndex.from_tuples(
    [("a", "foo"), ("a", "bar"), ("b", "foo"), ("b", "bah")], names=["lvl0", "lvl1"]
)


dfmi = (
    pd.DataFrame(
        np.arange(len(miindex) * len(micolumns)).reshape(
            (len(miindex), len(micolumns))
        ),
        index=miindex,
        columns=micolumns,
    )
    .sort_index()
    .sort_index(axis=1)
)

dfmi

lvl0           a         b     
lvl1         bar  foo  bah  foo
A0 B0 C0 D0    1    0    3    2
         D1    5    4    7    6
      C1 D0    9    8   11   10
         D1   13   12   15   14
      C2 D0   17   16   19   18
...          ...  ...  ...  ...
A3 B1 C1 D1  237  236  239  238
      C2 D0  241  240  243  242
         D1  245  244  247  246
      C3 D0  249  248  251  250
         D1  253  252  255  254

[64 rows x 4 columns]

In [24]:
# Basic Multiindex slicing using slices, lists, and labels.
dfmi.loc[(slice("A1", "A3"), slice(None), ["C1", "C3"]), :]

lvl0           a         b     
lvl1         bar  foo  bah  foo
A1 B0 C1 D0   73   72   75   74
         D1   77   76   79   78
      C3 D0   89   88   91   90
         D1   93   92   95   94
   B1 C1 D0  105  104  107  106
         D1  109  108  111  110
      C3 D0  121  120  123  122
         D1  125  124  127  126
A2 B0 C1 D0  137  136  139  138
         D1  141  140  143  142
      C3 D0  153  152  155  154
         D1  157  156  159  158
   B1 C1 D0  169  168  171  170
         D1  173  172  175  174
      C3 D0  185  184  187  186
         D1  189  188  191  190
A3 B0 C1 D0  201  200  203  202
         D1  205  204  207  206
      C3 D0  217  216  219  218
         D1  221  220  223  222
   B1 C1 D0  233  232  235  234
         D1  237  236  239  238
      C3 D0  249  248  251  250
         D1  253  252  255  254

In [25]:
# You can use pandas.IndexSlice to facilitate a more natural syntax using :, rather than using slice(None).
idx = pd.IndexSlice

dfmi.loc[idx[:, :, ['C1', 'C3']], idx[:, 'foo']]

lvl0           a    b
lvl1         foo  foo
A0 B0 C1 D0    8   10
         D1   12   14
      C3 D0   24   26
         D1   28   30
   B1 C1 D0   40   42
         D1   44   46
      C3 D0   56   58
         D1   60   62
A1 B0 C1 D0   72   74
         D1   76   78
      C3 D0   88   90
         D1   92   94
   B1 C1 D0  104  106
         D1  108  110
      C3 D0  120  122
         D1  124  126
A2 B0 C1 D0  136  138
         D1  140  142
      C3 D0  152  154
         D1  156  158
   B1 C1 D0  168  170
         D1  172  174
      C3 D0  184  186
         D1  188  190
A3 B0 C1 D0  200  202
         D1  204  206
      C3 D0  216  218
         D1  220  222
   B1 C1 D0  232  234
         D1  236  238
      C3 D0  248  250
         D1  252  254

In [26]:
#It is possible to perform quite complicated selections using this method on multiple axes at the same time.

dfmi.loc["A1", (slice(None), 'foo')]

lvl0        a    b
lvl1      foo  foo
B0 C0 D0   64   66
      D1   68   70
   C1 D0   72   74
      D1   76   78
   C2 D0   80   82
      D1   84   86
   C3 D0   88   90
      D1   92   94
B1 C0 D0   96   98
      D1  100  102
   C1 D0  104  106
      D1  108  110
   C2 D0  112  114
      D1  116  118
   C3 D0  120  122
      D1  124  126

In [27]:
dfmi.loc[idx[:, :, ["C1", "C3"]], idx[: ,"foo"]]

lvl0           a    b
lvl1         foo  foo
A0 B0 C1 D0    8   10
         D1   12   14
      C3 D0   24   26
         D1   28   30
   B1 C1 D0   40   42
         D1   44   46
      C3 D0   56   58
         D1   60   62
A1 B0 C1 D0   72   74
         D1   76   78
      C3 D0   88   90
         D1   92   94
   B1 C1 D0  104  106
         D1  108  110
      C3 D0  120  122
         D1  124  126
A2 B0 C1 D0  136  138
         D1  140  142
      C3 D0  152  154
         D1  156  158
   B1 C1 D0  168  170
         D1  172  174
      C3 D0  184  186
         D1  188  190
A3 B0 C1 D0  200  202
         D1  204  206
      C3 D0  216  218
         D1  220  222
   B1 C1 D0  232  234
         D1  236  238
      C3 D0  248  250
         D1  252  254

In [28]:
# Using a boolean indexer you can provide selection related to the values.

mask = dfmi[('a', 'foo')] > 200
dfmi.loc[idx[mask, :, ["C1", "C3"]], idx[:, "foo"]]

lvl0           a    b
lvl1         foo  foo
A3 B0 C1 D1  204  206
      C3 D0  216  218
         D1  220  222
   B1 C1 D0  232  234
         D1  236  238
      C3 D0  248  250
         D1  252  254

In [29]:
# You can also specify the axis argument to .loc to interpret the passed slicers on a single axis.
dfmi.loc(axis=0)[:, :, ["C1", "C3"]]

lvl0           a         b     
lvl1         bar  foo  bah  foo
A0 B0 C1 D0    9    8   11   10
         D1   13   12   15   14
      C3 D0   25   24   27   26
         D1   29   28   31   30
   B1 C1 D0   41   40   43   42
         D1   45   44   47   46
      C3 D0   57   56   59   58
         D1   61   60   63   62
A1 B0 C1 D0   73   72   75   74
         D1   77   76   79   78
      C3 D0   89   88   91   90
         D1   93   92   95   94
   B1 C1 D0  105  104  107  106
         D1  109  108  111  110
      C3 D0  121  120  123  122
         D1  125  124  127  126
A2 B0 C1 D0  137  136  139  138
         D1  141  140  143  142
      C3 D0  153  152  155  154
         D1  157  156  159  158
   B1 C1 D0  169  168  171  170
         D1  173  172  175  174
      C3 D0  185  184  187  186
         D1  189  188  191  190
A3 B0 C1 D0  201  200  203  202
         D1  205  204  207  206
      C3 D0  217  216  219  218
         D1  221  220  223  222
   B1 C1 D0  233  232  235  234
         D1  237  236  239  238
      C3 D0  249  248  251  250
         D1  253  252  255  254

In [30]:
# Furthermore, you can set the values using the following methods.
df2 = dfmi.copy()
df2.loc(axis = 0)[:, :, ["C1", "C3"]] = -10
df2

lvl0           a         b     
lvl1         bar  foo  bah  foo
A0 B0 C0 D0    1    0    3    2
         D1    5    4    7    6
      C1 D0  -10  -10  -10  -10
         D1  -10  -10  -10  -10
      C2 D0   17   16   19   18
...          ...  ...  ...  ...
A3 B1 C1 D1  -10  -10  -10  -10
      C2 D0  241  240  243  242
         D1  245  244  247  246
      C3 D0  -10  -10  -10  -10
         D1  -10  -10  -10  -10

[64 rows x 4 columns]

In [31]:
# Furthermore, you can set the values using the following methods.
df2 = dfmi.copy()
df2.loc[idx[:, :, ["C1", "C3"]], :] = df2 * 1000
df2

lvl0              a               b        
lvl1            bar     foo     bah     foo
A0 B0 C0 D0       1       0       3       2
         D1       5       4       7       6
      C1 D0    9000    8000   11000   10000
         D1   13000   12000   15000   14000
      C2 D0      17      16      19      18
...             ...     ...     ...     ...
A3 B1 C1 D1  237000  236000  239000  238000
      C2 D0     241     240     243     242
         D1     245     244     247     246
      C3 D0  249000  248000  251000  250000
         D1  253000  252000  255000  254000

[64 rows x 4 columns]

### Cross-Section
The `xs()` method of DataFrame additionally takes a level argument to make selecting data at a particular level of a `MultiIndex` easier.

In [32]:
df = df.T
df

A         B         C
first second                              
bar   one     1.298022 -0.220813  0.830147
      two     0.227811  1.271892 -1.181357
baz   one     0.158003  0.158113 -0.015977
      two     0.940212 -0.812318  1.011322
foo   one    -0.231041  1.888800 -1.048584
      two    -1.076115 -0.243946 -0.490976
qux   one    -0.014290 -1.414743 -0.736137
      two    -0.373190  1.701547  1.297449

In [33]:
df.xs('one', level = 'second') # Give me every row where the second level of the index is equal to "one"

,A,B,C
first,,,
bar,1.298022,-0.220813,0.830147
baz,0.158003,0.158113,-0.015977
foo,-0.231041,1.888800,-1.048584
qux,-0.014290,-1.414743,-0.736137


In [34]:
# using the slicers
df.loc[(slice(None), "one"), :]

,,A,B,C
first,second,,,
bar,one,1.298022,-0.220813,0.830147
baz,one,0.158003,0.158113,-0.015977
foo,one,-0.231041,1.888800,-1.048584
qux,one,-0.014290,-1.414743,-0.736137


In [35]:
# You can also select on the columns with xs, by providing the axis argument.
df = df.T
df.xs("one", level = "second", axis = 1)

first,bar,baz,foo,qux
A,1.298022,0.158003,-0.231041,-0.014290
B,-0.220813,0.158113,1.888800,-1.414743
C,0.830147,-0.015977,-1.048584,-0.736137


In [36]:
# using the slicers
df.loc[:, (slice(None), "one")]

first,bar,baz,foo,qux
second,one,one,one,one
A,1.298022,0.158003,-0.231041,-0.014290
B,-0.220813,0.158113,1.888800,-1.414743
C,0.830147,-0.015977,-1.048584,-0.736137


In [37]:
# xs also allows selection with multiple keys.
df.xs(("one", "bar"), level = ("second", "first"), axis = 1)

first,bar
second,one
A,1.298022
B,-0.220813
C,0.830147


In [38]:
# using the slicers
df.loc[:, ("bar", "one")]

A    1.298022
B   -0.220813
C    0.830147
Name: (bar, one), dtype: float64

In [39]:
df

first        bar                 baz                 foo                 qux  \
second       one       two       one       two       one       two       one   
A       1.298022  0.227811  0.158003  0.940212 -0.231041 -1.076115 -0.014290   
B      -0.220813  1.271892  0.158113 -0.812318  1.888800 -0.243946 -1.414743   
C       0.830147 -1.181357 -0.015977  1.011322 -1.048584 -0.490976 -0.736137   

first             
second       two  
A      -0.373190  
B       1.701547  
C       1.297449

In [40]:
# You can pass drop_level=False to xs to retain the level that was selected.
df.xs("one", level = "second", axis = 1, drop_level = False)

first,bar,baz,foo,qux
second,one,one,one,one
A,1.298022,0.158003,-0.231041,-0.014290
B,-0.220813,0.158113,1.888800,-1.414743
C,0.830147,-0.015977,-1.048584,-0.736137


In [41]:
# compare the above with the result using drop_level=True (the default value)
df.xs("one", level = "second", axis = 1, drop_level = True)

first,bar,baz,foo,qux
A,1.298022,0.158003,-0.231041,-0.014290
B,-0.220813,0.158113,1.888800,-1.414743
C,0.830147,-0.015977,-1.048584,-0.736137


### Swapping levels with swaplevel
The `swaplevel()` method can switch the order of two levels:

In [42]:
midx = pd.MultiIndex(
    levels=[["zero", "one"], ["x", "y"]], codes=[[1, 1, 0, 0], [1, 0, 1, 0]]
)


df = pd.DataFrame(np.random.randn(4, 2), index=midx)

df[:5]

0         1
one  y -0.742717 -0.912138
     x -2.159573  0.821666
zero y -0.477923  0.945425
     x -1.371552  0.901811

In [43]:
df[:5].swaplevel(0, 1, axis = 0)

,,0,1
y,one,-0.742717,-0.912138
x,one,-2.159573,0.821666
y,zero,-0.477923,0.945425
x,zero,-1.371552,0.901811


### Sorting a `MultiIndex`
For **`MultiIndex`**-ed objects to be indexed and sliced effectively, they need to be sorted. As with any index, you can use **`sort_index()`**.


In [44]:
tuples

[('bar', 'one'),
 ('bar', 'two'),
 ('baz', 'one'),
 ('baz', 'two'),
 ('foo', 'one'),
 ('foo', 'two'),
 ('qux', 'one'),
 ('qux', 'two')]

In [45]:
import random
random.shuffle(tuples)
s = pd.Series(np.random.randn(8), index = pd.MultiIndex.from_tuples(tuples))
s

foo  one    0.917849
baz  two    0.396087
     one    0.433144
qux  two    0.895103
foo  two    0.630010
bar  two   -0.479218
     one   -0.049878
qux  one    0.520391
dtype: float64

In [46]:
s.sort_index()

bar  one   -0.049878
     two   -0.479218
baz  one    0.433144
     two    0.396087
foo  one    0.917849
     two    0.630010
qux  one    0.520391
     two    0.895103
dtype: float64

In [47]:
s.sort_index(level = 0)

bar  one   -0.049878
     two   -0.479218
baz  one    0.433144
     two    0.396087
foo  one    0.917849
     two    0.630010
qux  one    0.520391
     two    0.895103
dtype: float64

In [48]:
s.sort_index(level=1)

bar  one   -0.049878
baz  one    0.433144
foo  one    0.917849
qux  one    0.520391
bar  two   -0.479218
baz  two    0.396087
foo  two    0.630010
qux  two    0.895103
dtype: float64

In [49]:
# You may also pass a level name to sort_index if the MultiIndex levels are named.
s.index = s.index.set_names(["L1", "L2"])
s

L1   L2 
foo  one    0.917849
baz  two    0.396087
     one    0.433144
qux  two    0.895103
foo  two    0.630010
bar  two   -0.479218
     one   -0.049878
qux  one    0.520391
dtype: float64

In [50]:
s.sort_index(level = "L1")

L1   L2 
bar  one   -0.049878
     two   -0.479218
baz  one    0.433144
     two    0.396087
foo  one    0.917849
     two    0.630010
qux  one    0.520391
     two    0.895103
dtype: float64

In [51]:
s.sort_index(level="L2")

L1   L2 
bar  one   -0.049878
baz  one    0.433144
foo  one    0.917849
qux  one    0.520391
bar  two   -0.479218
baz  two    0.396087
foo  two    0.630010
qux  two    0.895103
dtype: float64

In [52]:
df.T

one                zero          
          y         x         y         x
0 -0.742717 -2.159573 -0.477923 -1.371552
1 -0.912138  0.821666  0.945425  0.901811

In [53]:
# On higher dimensional objects, you can sort any of the other axes by level if they have a MultiIndex:
df.T.sort_index(level=1, axis=1)

,one,zero,one,zero
,x,x,y,y
0,-2.159573,-1.371552,-0.742717,-0.477923
1,0.821666,0.901811,-0.912138,0.945425


In [54]:
# Indexing will work even if the data are not sorted, but will be rather inefficient (and show a PerformanceWarning). It will also return a copy of the data rather than a view:
dfm = pd.DataFrame(
    {"jim": [0, 0, 1, 1], "joe": ["x", "x", "z", "y"], "jolie": np.random.rand(4)}
)
dfm = dfm.set_index(["jim", "joe"])
dfm

jolie
jim joe          
0   x    0.316006
    x    0.938484
1   z    0.147855
    y    0.982748

In [55]:
dfm.loc[(1, 'z')]

C:\Users\yugal\AppData\Local\Temp\ipykernel_7136\1188037207.py:1: PerformanceWarning: indexing past lexsort depth may impact performance.
  dfm.loc[(1, 'z')]


,,jolie
jim,joe,
1,z,0.147855


In [56]:
# Furthermore, if you try to index something that is not fully lexsorted, this can raise:
dfm.loc[(0, 'y'):(1, 'z')]

UnsortedIndexError: 'Key length (2) was greater than MultiIndex lexsort depth (1)'

In [57]:
# The is_monotonic_increasing() method on a MultiIndex shows if the index is sorted:
dfm.index.is_monotonic_increasing

False

In [58]:
dfm = dfm.sort_index()
dfm.index.is_monotonic_increasing

True

In [59]:
# And now selection works as expected.
dfm.loc[(0, "y"):(1, "z")]

jolie
jim joe          
1   y    0.982748
    z    0.147855

#### Binning data with `cut` and `qcut`

**`cut()`** and **`qcut()`** both return a `Categorical` object, and the bins they create are stored as an `IntervalIndex` in its `.categories` attribute.


In [60]:
c = pd.cut(range(4), bins = 2)
c
# range(4) → [0, 1, 2, 3]
# It splits them into 2 equal-width bins:
# 0 and 1 go into the first bin (-0.003, 1.5]
# 2 and 3 go into the second bin (1.5, 3.0]

[(-0.003, 1.5], (-0.003, 1.5], (1.5, 3.0], (1.5, 3.0]]
Categories (2, interval[float64, right]): [(-0.003, 1.5] < (1.5, 3.0]]

**`cut()`** also accepts an `IntervalIndex` for its `bins` argument, which enables a useful pandas idiom. First, We call **`cut()`** with some data and `bins` set to a fixed number, to generate the bins. Then, we pass the values of `.categories` as the `bins` argument in subsequent calls to **`cut()`**, supplying new data which will be binned into the same bins.


In [61]:
pd.cut([0, 3, 5, 1], bins=c.categories)

# 0 → first bin
# 3 → second bin
# 5 → outside all bins → NaN
# 1 → first bin

[(-0.003, 1.5], (1.5, 3.0], NaN, (-0.003, 1.5]]
Categories (2, interval[float64, right]): [(-0.003, 1.5] < (1.5, 3.0]]